# Musical Bingo — Initial Analysis

This notebook is a starting point for analysing the musical bingo JSON generated from the PDF.

It covers:
- loading and validating the dataset
- basic statistics
- song and artist frequencies
- repeated songs
- sheet-level exploration
- preparation for comparing bingo sheets against randomly ordered music playlists

The dataset contains **150 bingo sheets**, with **15 songs per sheet**.

In [10]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

JSON_PATH = Path("extract_chatgpt.json")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Source file: {data['source_file']}")
print(f"PDF pages: {data['total_pdf_pages']}")
print(f"Sheets: {data['total_sheets']}")
print(f"Songs per sheet: {data['songs_per_sheet']}")
print(f"Groups per sheet: {data['groups_per_sheet']}")
print(f"Songs per group: {data['songs_per_group']}")

Source file: Cartones_Bingo_Musical_A4.pdf
PDF pages: 75
Sheets: 150
Songs per sheet: 15
Groups per sheet: 3
Songs per group: 5


## 1. Validate the dataset

In [12]:
assert len(data['sheets']) == data['total_sheets']

assert all(
    len(group) == data["songs_per_group"]
    for sheet in data["sheets"]
    for group in sheet["songs"]
)

assert all(
    'artist' in song and 'title' in song
    for sheet in data['sheets']
    for group in sheet['songs']
    for song in group
)

print('Validation successful.')
print(
    f"Total song entries: {sum(len(group) for s in data['sheets'] for group in s['songs']):,}"
)

Validation successful.
Total song entries: 2,250


## 2. Convert the JSON into a convenient DataFrame

In [24]:
rows = []

for sheet in data['sheets']:
    for row_id, row in enumerate(sheet['songs'], start=1):
        for col_id, song in enumerate(row, start=1):
            rows.append(
                {
                    "sheet_id": sheet["sheet_id"],
                    "pdf_page": sheet["pdf_page"],
                    "sheet_on_page": sheet["sheet_on_page"],
                    "row": row_id,
                    "col": col_id,
                    "artist": song["artist"],
                    "title": song["title"],
                    "song": f"{song['artist']} — {song['title']}",
                }
            )

songs_df = pd.DataFrame(rows)
songs_df.head(20)

,sheet_id,pdf_page,sheet_on_page,row,col,artist,title,song
0,1,1,1,1,1,Las Ketchup,Aserejé,Las Ketchup — Aserejé
1,1,1,1,1,2,The Cranberries,Zombie,The Cranberries — Zombie
2,1,1,1,1,3,Spice Girls,Wannabe,Spice Girls — Wannabe
3,1,1,1,1,4,A-ha,Take on Me,A-ha — Take on Me
4,1,1,1,1,5,Carlos Vives,Volvi a Nacer,Carlos Vives — Volvi a Nacer
5,1,1,1,2,1,Camela,Cuando Zarpa el Amor,Camela — Cuando Zarpa el Amor
6,1,1,1,2,2,Sau,Boig per Tu,Sau — Boig per Tu
7,1,1,1,2,3,Hemendik At,Eskutitza,Hemendik At — Eskutitza
8,1,1,1,2,4,La Fumiga,Ja Dormiré,La Fumiga — Ja Dormiré
9,1,1,1,2,5,Huntza,Aldapan Gora,Huntza — Aldapan Gora


## 3. Basic statistics

In [25]:
total_entries = len(songs_df)
unique_songs = songs_df['song'].nunique()
unique_artists = songs_df['artist'].nunique()

stats = pd.Series({
    'Bingo sheets': songs_df['sheet_id'].nunique(),
    'Songs per sheet': songs_df.groupby('sheet_id').size().iloc[0],
    'Total song entries': total_entries,
    'Unique songs': unique_songs,
    'Unique artists': unique_artists,
    'Average appearances per unique song': total_entries / unique_songs,
})

stats

Bingo sheets                            150.000000
Songs per sheet                          15.000000
Total song entries                     2250.000000
Unique songs                             95.000000
Unique artists                           94.000000
Average appearances per unique song      23.684211
dtype: float64

## 4. Most frequently appearing songs

In [26]:
song_frequency = (
    songs_df.groupby(['artist', 'title'], as_index=False)
    .size()
    .rename(columns={'size': 'appearances'})
    .sort_values('appearances', ascending=False)
)

song_frequency['song'] = song_frequency['artist'] + ' — ' + song_frequency['title']
song_frequency.head(30)

,artist,title,appearances,song
73,Sau,Boig per Tu,36,Sau — Boig per Tu
3,Alaska y Dinarama,Ni Tu Ni Nadie,34,Alaska y Dinarama — Ni Tu Ni Nadie
58,Mago de Oz,Fiesta Pagana,32,Mago de Oz — Fiesta Pagana
90,Whitney Houston,I Wanna Dance with Somebody,32,Whitney Houston — I Wanna Dance with Somebody
92,Zea Mays,Negua Joan Da Ta,31,Zea Mays — Negua Joan Da Ta
18,Celia Cruz,La Vida Es un Carnaval,31,Celia Cruz — La Vida Es un Carnaval
0,A-ha,Take on Me,30,A-ha — Take on Me
49,La Fumiga,Ja Dormiré,30,La Fumiga — Ja Dormiré
57,Luis Fonsi,Despacito,30,Luis Fonsi — Despacito
88,Vengaboys,Boom Boom Boom Boom,30,Vengaboys — Boom Boom Boom Boom


In [ ]:
top_n = 20

plot_df = song_frequency.head(top_n).sort_values('appearances')

plt.figure(figsize=(10, 8))
plt.barh(plot_df['song'], plot_df['appearances'])
plt.xlabel('Number of bingo sheets')
plt.title(f'Top {top_n} most frequent songs')
plt.tight_layout()
plt.show()

## 5. Most frequent artists

In [27]:
artist_frequency = (
    songs_df.groupby('artist')
    .agg(
        appearances=('song', 'size'),
        unique_songs=('song', 'nunique')
    )
    .sort_values(['appearances', 'unique_songs'], ascending=False)
)

artist_frequency.head(30)

,appearances,unique_songs
artist,,
Sau,36,1
Alaska y Dinarama,34,1
Hertzainak,33,2
Mago de Oz,32,1
Whitney Houston,32,1
Celia Cruz,31,1
Zea Mays,31,1
A-ha,30,1
La Fumiga,30,1


## 6. Repeated songs and distribution

In [28]:
frequency_distribution = song_frequency['appearances'].value_counts().sort_index()

print('How many unique songs appear N times:')
display(frequency_distribution.rename('unique_songs'))

print('\nSongs appearing more than once:')
display(song_frequency[song_frequency['appearances'] > 1].reset_index(drop=True))

How many unique songs appear N times:


appearances
12    2
14    2
15    3
16    2
17    3
18    4
19    3
20    6
21    4
22    8
23    6
24    8
25    8
26    5
27    8
28    8
29    5
30    4
31    2
32    2
34    1
36    1
Name: unique_songs, dtype: int64


Songs appearing more than once:


,artist,title,appearances,song
0,Sau,Boig per Tu,36,Sau — Boig per Tu
1,Alaska y Dinarama,Ni Tu Ni Nadie,34,Alaska y Dinarama — Ni Tu Ni Nadie
2,Mago de Oz,Fiesta Pagana,32,Mago de Oz — Fiesta Pagana
3,Whitney Houston,I Wanna Dance with Somebody,32,Whitney Houston — I Wanna Dance with Somebody
4,Zea Mays,Negua Joan Da Ta,31,Zea Mays — Negua Joan Da Ta
...,...,...,...,...
90,En Tol Sarmiento & Buhos,T'he Trobat a Faltar,15,En Tol Sarmiento & Buhos — T'he Trobat a Faltar
91,Hertzainak,Pakean Utzi Arte,14,Hertzainak — Pakean Utzi Arte
92,Beth,Dime,14,Beth — Dime
93,Don Omar,Dale Don Dale,12,Don Omar — Dale Don Dale


In [29]:
least_used_count = song_frequency["appearances"].min()
least_used_songs = (
    song_frequency.loc[song_frequency["appearances"] == least_used_count, ["artist", "title", "appearances"]]
    .sort_values(["artist", "title"])
    .reset_index(drop=True)
 )

print(f"Least-used songs appear on {least_used_count} bingo sheets:")
least_used_songs

Least-used songs appear on 12 bingo sheets:


,artist,title,appearances
0,Don Omar,Dale Don Dale,12
1,Tina Turner,The Best,12


## 7. Inspect an individual bingo sheet

In [31]:
SHEET_ID = 1

sheet_view = songs_df[songs_df["sheet_id"] == SHEET_ID].copy()
sheet_view[["row", "col", "artist", "title"]]

,row,col,artist,title
0,1,1,Las Ketchup,Aserejé
1,1,2,The Cranberries,Zombie
2,1,3,Spice Girls,Wannabe
3,1,4,A-ha,Take on Me
4,1,5,Carlos Vives,Volvi a Nacer
5,2,1,Camela,Cuando Zarpa el Amor
6,2,2,Sau,Boig per Tu
7,2,3,Hemendik At,Eskutitza
8,2,4,La Fumiga,Ja Dormiré
9,2,5,Huntza,Aldapan Gora


## 8. Compare overlap between bingo sheets

In [32]:
sheet_sets = {
    sheet_id: set(group['song'])
    for sheet_id, group in songs_df.groupby('sheet_id')
}

def sheet_overlap(sheet_a, sheet_b):
    return len(sheet_sets[sheet_a] & sheet_sets[sheet_b])

# Example: overlap of sheet 1 with every other sheet
reference_sheet = 1
overlaps = pd.Series({
    sheet_id: sheet_overlap(reference_sheet, sheet_id)
    for sheet_id in sheet_sets
    if sheet_id != reference_sheet
}).sort_values(ascending=False)

overlaps.head(20)

115    8
43     6
141    6
53     6
41     5
34     5
23     5
51     5
94     5
128    5
74     5
99     5
36     4
33     4
54     4
45     4
5      4
19     4
13     4
117    4
dtype: int64

## 9. Prepare for playlist / bingo-duration analysis

The next step can be to provide one or more randomly ordered playlists and simulate playback.

For each playlist we can determine, for every bingo sheet:
- the position at which each of its 15 songs is played
- the position of the 5th, 10th, 15th, etc. matched song
- the first position at which the sheet reaches a bingo condition
- how many songs have been played when the first/last sheet wins
- how many sheets are still incomplete after a given number of played songs

The exact definition of **bingo** (for example, any 5 songs, a row, a column, or a specific layout) should be encoded explicitly before running the simulation.

In [33]:
# Example helper: return the playback position of each song belonging to a sheet.
# `playlist` should be a list of song strings in playback order, using the same
# 'Artist — Title' representation as songs_df['song'].

def sheet_match_positions(sheet_id, playlist):
    target_songs = sheet_sets[sheet_id]
    return {
        song: position
        for position, song in enumerate(playlist, start=1)
        if song in target_songs
    }

# Example usage:
# playlist = songs_df['song'].drop_duplicates().sample(frac=1, random_state=42).tolist()
# positions = sheet_match_positions(1, playlist)
# positions

## 10. Useful next analyses

Potential extensions:

1. Import several randomly sorted playlists.
2. Normalize song names to handle spelling/capitalization differences.
3. Match playlist entries to bingo songs.
4. Simulate the playback of each playlist.
5. Implement the actual bingo-card winning rules.
6. Calculate the distribution of game duration across playlists and sheets.
7. Identify songs that strongly affect the time-to-bingo.
8. Test whether the 150 cards provide sufficiently balanced coverage of the song pool.

In [34]:
song_list_filename = "Bingo_Musical_Boda_EstiCarles.xlsx"
song_list_file = pd.ExcelFile(song_list_filename)
song_list_df = pd.read_excel(
    song_list_filename, sheet_name="Playlist", header=None, nrows=8
)
print(song_list_df.to_string(index=False, header=False))

Nº                    Categoria      Artista                     Cancion
 1 Clasicos Internacionales 80s  Tina Turner                    The Best
 2 Clasicos Internacionales 80s Bonnie Tyler  Total Eclipse of the Heart
 3 Clasicos Internacionales 80s   Alphaville               Forever Young
 4 Clasicos Internacionales 80s Cyndi Lauper Girls Just Want to Have Fun
 5 Clasicos Internacionales 80s         Cher                     Believe
 6 Clasicos Internacionales 80s      Roxette      It Must Have Been Love
 7 Clasicos Internacionales 80s        Queen           Don't Stop Me Now


In [35]:
import re
import unicodedata

EXCEL_PATH = Path("Bingo_Musical_Boda_EstiCarles.xlsx")
excel_sheets = pd.read_excel(EXCEL_PATH, sheet_name=None)


def normalize_label(value):
    value = unicodedata.normalize("NFKD", str(value).casefold())
    value = "".join(char for char in value if not unicodedata.combining(char))
    return re.sub(r"[^a-z0-9]+", "", value)


artist_labels = {"artist", "artista", "interprete", "interpretes", "grupo", "band"}
title_labels = {"title", "titulo", "cancion", "song", "tema", "track"}

excel_sheet_name = None
for sheet_name, sheet_df in excel_sheets.items():
    columns_by_label = {normalize_label(column): column for column in sheet_df.columns}
    artist_column = next(
        (
            columns_by_label[label]
            for label in artist_labels
            if label in columns_by_label
        ),
        None,
    )
    title_column = next(
        (
            columns_by_label[label]
            for label in title_labels
            if label in columns_by_label
        ),
        None,
    )
    if artist_column is not None and title_column is not None:
        excel_sheet_name = sheet_name
        excel_song_rows = sheet_df[[artist_column, title_column]].copy()
        excel_song_rows.columns = ["artist", "title"]
        break

if excel_sheet_name is None:
    available_columns = {
        name: list(frame.columns) for name, frame in excel_sheets.items()
    }
    raise ValueError(
        "Could not find artist/title columns in the Excel workbook. "
        f"Available columns by sheet: {available_columns}"
    )

excel_song_rows = excel_song_rows.dropna(subset=["artist", "title"]).copy()
excel_song_rows["artist"] = excel_song_rows["artist"].astype(str).str.strip()
excel_song_rows["title"] = excel_song_rows["title"].astype(str).str.strip()


def song_key(artist, title):
    return normalize_label(artist), normalize_label(title)


bingo_song_rows = songs_df[["artist", "title"]].drop_duplicates().copy()
bingo_song_rows["match_key"] = [
    song_key(artist, title)
    for artist, title in zip(bingo_song_rows["artist"], bingo_song_rows["title"])
]
excel_song_rows["match_key"] = [
    song_key(artist, title)
    for artist, title in zip(excel_song_rows["artist"], excel_song_rows["title"])
]

excel_keys = set(excel_song_rows["match_key"])
bingo_keys = set(bingo_song_rows["match_key"])
missing_from_excel = bingo_song_rows[~bingo_song_rows["match_key"].isin(excel_keys)][
    ["artist", "title"]
].reset_index(drop=True)
extra_in_excel = (
    excel_song_rows[~excel_song_rows["match_key"].isin(bingo_keys)][["artist", "title"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Imported {len(excel_song_rows)} song rows from '{EXCEL_PATH.name}', sheet '{excel_sheet_name}'."
)
print(f"Unique songs on bingo sheets: {len(bingo_keys)}")
print(f"Unique songs in Excel: {len(excel_keys)}")
print(f"Songs from bingo sheets missing in Excel: {len(missing_from_excel)}")
print(f"Songs in Excel not found on bingo sheets: {len(extra_in_excel)}")
print(f"Validation: {'PASS' if missing_from_excel.empty else 'FAIL'}")

if not missing_from_excel.empty:
    display(missing_from_excel)
if not extra_in_excel.empty:
    print("Excel-only songs:")
    display(extra_in_excel)

Imported 95 song rows from 'Bingo_Musical_Boda_EstiCarles.xlsx', sheet 'Playlist'.
Unique songs on bingo sheets: 95
Unique songs in Excel: 95
Songs from bingo sheets missing in Excel: 0
Songs in Excel not found on bingo sheets: 0
Validation: PASS
